# WKNN

WKNN(_Weighted K-Nearest Neighbor_) adalah pengembangan dari algoritma _K-Nearest Neighbors_ (KNN) yang memberikan bobot berbeda pada setiap tetangga terdekat.

Imputasi menggunakan WKNN digunakan untuk mengisi nilai yang hilang (missing value) pada dataset dengan memanfaatkan data tetangga terdekat dan memberikan bobot berdasarkan jarak.

In [2]:
import pandas as pd

df = pd.read_csv("../data.csv")

print("Contoh Data:")
display(df)

Contoh Data:


,No,IPK,PO,JML
0,1,2,2000000,2.0
1,2,3,3000000,3.0
2,3,4,2000000,2.0
3,4,2,2000000,3.0
4,5,3,3000000,2.0
5,6,4,4000000,3.0
6,7,2,3000000,NaN


## Normalisasi

Pada dataset tersebut, terdapat satu buah _missing value_ pada objek ke-7 kolom JML. Sebelum melakukan imputasi menggunakan WKNN, data terlebih dahulu dinormalisasi. Pada kasus ini, saya menggunakan metode normalisasi `Min-Max Normalization` sehingga didapatkan tabel seperti berikut

In [6]:
import pandas as pd

df = pd.read_csv("../data.csv")

kolom = ['IPK','PO','JML']

for col in kolom:
    min_val = df[col].min()
    max_val = df[col].max()
    df[col] = (df[col] - min_val) / (max_val - min_val)
display(df)

,No,IPK,PO,JML
0,1,0.0,0.0,0.0
1,2,0.5,0.5,1.0
2,3,1.0,0.0,0.0
3,4,0.0,0.0,1.0
4,5,0.5,0.5,0.0
5,6,1.0,1.0,1.0
6,7,0.0,0.5,NaN


## Kemiripan (Similarity)

Setelah data dinormalisasi, langkah selanjutnya adalah menghitung *kemiripan* (*similarity*) antar data.  

Perhitungan ini digunakan untuk menentukan kedekatan antara data yang memiliki *missing value* dengan data lainnya (tetangga terdekat).

Rumus yang digunakan adalah sebagai berikut:

$$
\frac{1}{s_i} = \sum_{h_i \in O_i \cap O_j} (y_{ih} - y_{jh})^2
$$

### Keterangan

- $(y_{ih} - y_{jh})^2$ adalah kuadrat selisih nilai atribut antara data ke-$i$ dan data ke-$j$ (mengacu pada konsep *Euclidean Distance*).

- $O_i \cap O_j$ adalah irisan atribut yang tersedia pada kedua data (hanya atribut yang tidak memiliki *missing value* yang dihitung).

- $\frac{1}{s_i}$ adalah nilai jarak. Semakin kecil nilai jarak, maka semakin besar nilai $s_i$, sehingga kedua data semakin mirip.

## Contoh Perhitungan

Menghitung kemiripan antara data ke-7 (sebagai data target) dengan data lainnya.

### 1. Dengan Data ke-1

$$
\frac{1}{s_1} = (0 - 0)^2 + (0.5 - 0)^2
$$

$$
= 0 + 0.25 = 0.25
$$

$$
s_1 = \frac{1}{0.25} = 4
$$

### 2. Dengan Data ke-2

$$
\frac{1}{s_2} = (0 - 0.5)^2 + (0.5 - 0.5)^2
$$

$$
= 0.25 + 0 = 0.25
$$

$$
s_2 = \frac{1}{0.25} = 4
$$

## Hasil Perhitungan

Dengan cara yang sama, dilakukan perhitungan terhadap data ke-3 hingga data ke-6, sehingga diperoleh hasil sebagai berikut:

$$
\begin{aligned}
s_1 &= 4 \\
s_2 &= 4 \\
s_3 &= 0.8 \\
s_4 &= 4 \\
s_5 &= 4 \\
s_6 &= 0.8
\end{aligned}
$$

## Kesimpulan

- Nilai $s$ yang lebih besar menunjukkan tingkat kemiripan yang lebih tinggi.  
- Data dengan nilai kemiripan terbesar dapat digunakan sebagai acuan untuk mengisi *missing value*.  

Pada kasus ini, data yang paling mirip dengan data ke-7 adalah:

- Data ke-1  
- Data ke-2  
- Data ke-4  
- Data ke-5  

karena memiliki nilai $s = 4$.

## Imputasi

Setelah mendapatkan bobot($s_i$) untuk setiap tetangga, kita gunakan rumus berikut untuk mengisi nilai yang kosong.

$$
\hat{y}_{ih} = \frac{\sum_{j \in I_{Kih}} s_i(y_j) y_{jh}}{\sum_{j \in I_{Kih}} s_i(y_j)}
$$

Dimana

- $\hat{y}_{ih}$: Nilai hasil prediksi (imputasi) untuk baris $i$ pada kolom $h$.
- $I_{Kih}$: Himpunan $K$ tetangga terdekat yang memiliki data pada kolom $h$.
- $s_i(y_j) y_{jh}$: Nilai dari tetangga ke-$j$ dikalikan dengan bobotnya. Ini memastikan tetangga yang paling mirip memberikan kontribusi lebih besar.
- Pembagi ($\sum s_i$): Digunakan untuk menormalisasi bobot sehingga hasil akhirnya tetap berada dalam skala yang wajar (rata-rata tertimbang).

Rumus tersebut apabila diterapkan pada dataset diatas, maka akan didapatkan hasil sebagai berikut


\begin{align*}
\hat{y}_{ih} &= \frac{(0 \times 4) + (1 \times 4) + (0 \times 0.8) + (1 \times 4) + (0 \times 4) + (1 \times 0.8)}{4 + 4 + 0.8 + 4 + 4 + 0.8} \\
&= \frac{0 + 4 + 0 + 4 + 0 + 0.8}{17.6} \\
&= \frac{8.8}{17.6} \\
&= 0.5
\end{align*}


Jadi nilai prediksi missing value menggunakan WKNN adalah `0.5`, perhitungan diatas apabila diterapkan pada Python akan menghasilkan sebagai berikut

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor as KNR

df = pd.read_csv("../data.csv")
df = pd.DataFrame(df)

df_train = df.iloc[:6].copy()

df_test = df.iloc[6:].copy()

scaler_features = MinMaxScaler()
scaler_target = MinMaxScaler()

train_X = scaler_features.fit_transform(df_train[['IPK', 'PO']])
test_X = scaler_features.transform(df_test[['IPK', 'PO']])

train_y = scaler_target.fit_transform(df_train[['JML']])

def custom_weights(distances):
    return 1 / (distances**2)

knn = KNR(n_neighbors=6, weights=custom_weights, metric='euclidean')
knn.fit(train_X, train_y)

pred_scaled = knn.predict(test_X)

pred_final = scaler_target.inverse_transform(pred_scaled)

print(f"Prediksi JML (Ternormalisasi): {pred_scaled[0][0]}")

Prediksi JML (Ternormalisasi): 0.5
